# Google public GA4 sample: descriptive findings

This notebook runs offline from committed aggregate-only artifacts retrieved from `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*` for 2020-11-01 through 2021-01-31. It contains no synthetic records or event-level data. Session channels use the first paired event-scoped `source`/`medium`, never `traffic_source` (which is first-user acquisition). Attribution is descriptive, not causal.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from marketing_measurement.analysis import (
    add_retention_rate,
    attribution_credits,
    funnel_rates,
)

OBSERVED = ROOT / 'data' / 'observed' / 'ga4_public_sample'
DERIVED = ROOT / 'data' / 'derived' / 'ga4_public_sample'
DERIVED.mkdir(parents=True, exist_ok=True)
metadata = json.loads((OBSERVED / 'retrieval_metadata.json').read_text())
funnel = pd.read_json(OBSERVED / 'funnel_daily_by_channel.json')
cohorts = pd.read_json(OBSERVED / 'cohort_retention.json')
paths = pd.read_json(OBSERVED / 'conversion_channel_paths.json')
parameter_availability = pd.read_json(OBSERVED / 'event_parameter_availability.json')

funnel['session_start_date'] = pd.to_datetime(funnel['session_start_date'])
assert metadata['project_id'] == 'christina-data-portfolio-2026'
assert metadata['maximum_bytes_billed'] == 4_000_000_000
assert funnel['session_start_date'].min() == pd.Timestamp('2020-11-01')
assert funnel['session_start_date'].max() == pd.Timestamp('2021-01-31')
assert {'user_pseudo_id', 'ga_session_id', 'transaction_id'}.isdisjoint(paths.columns)


In [2]:
funnel_totals = funnel[['views', 'engaged_sessions', 'add_to_carts', 'checkouts', 'purchases']].sum().to_frame().T
funnel_summary = funnel_rates(funnel_totals).iloc[0]

cohorts['cohort_date'] = pd.to_datetime(cohorts['cohort_date'])
cohorts = add_retention_rate(cohorts)
cohort_sizes = cohorts.loc[cohorts['days_since_acquisition'].eq(0)].set_index('cohort_date')['cohort_users']
day_seven = cohorts.loc[cohorts['days_since_acquisition'].eq(7)].set_index('cohort_date')['retained_users']
day_seven = day_seven.reindex(cohort_sizes.index, fill_value=0)
eligible_users = int(cohort_sizes.sum())
day_seven_users = int(day_seven.sum())

paths['conversion_date'] = pd.to_datetime(paths['conversion_date'])
paths['lookback_start_date'] = pd.to_datetime(paths['lookback_start_date'])
assert paths['conversion_date'].min() >= pd.Timestamp('2020-12-01')
assert (paths['conversion_date'] - paths['lookback_start_date']).eq(pd.Timedelta(days=30)).all()
assert paths['eligible_30_day_lookback'].astype(str).eq('true').all()

path_credit_rows = []
for path_index, path in paths.iterrows():
    channels = str(path['channel_path']).split(' > ')
    path_credit_rows.extend(
        {'conversion_id': f'observed_path_{path_index}', 'channel': channel, 'touch_number': touch, 'conversion_weight': int(path['converted_sessions'])}
        for touch, channel in enumerate(channels, start=1)
    )
path_frame = pd.DataFrame(path_credit_rows)
attribution_summary = {}
attribution_totals = {}
for model in ('first_touch', 'last_touch', 'linear', 'time_decay'):
    credits = attribution_credits(path_frame, model)
    credits['weighted_credit'] = credits['credit'] * credits['conversion_weight']
    attribution_summary[model] = (
        credits.groupby('channel')['weighted_credit'].sum().sort_values(ascending=False).round(6).to_dict()
    )
    attribution_totals[model] = round(float(credits['weighted_credit'].sum()), 6)
    assert attribution_totals[model] == float(paths['converted_sessions'].sum())

summary = {
    'source': {
        'dataset': 'bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*',
        'coverage_start': '2020-11-01',
        'coverage_end': '2021-01-31',
        'retrieved_at_utc': metadata['retrieved_at_utc'],
        'session_channel_definition': 'first paired event-scoped source/medium in each session',
    },
    'funnel': {
        'views': int(funnel_summary['views']),
        'engaged_sessions': int(funnel_summary['engaged_sessions']),
        'add_to_carts': int(funnel_summary['add_to_carts']),
        'checkouts': int(funnel_summary['checkouts']),
        'purchases': int(funnel_summary['purchases']),
        'engagement_rate': round(float(funnel_summary['engaged_sessions_rate']), 6),
        'add_to_cart_rate': round(float(funnel_summary['add_to_carts_rate']), 6),
        'checkout_rate': round(float(funnel_summary['checkouts_rate']), 6),
        'purchase_rate': round(float(funnel_summary['purchases_rate']), 6),
    },
    'day_7_retention': {
        'eligible_cohorts': len(cohort_sizes),
        'cohort_users': eligible_users,
        'retained_users': day_seven_users,
        'rate': round(day_seven_users / eligible_users, 6),
        'definition': 'user_first_touch_timestamp within source window; day-7 complete',
    },
    'attribution': {
        'eligible_converted_sessions': int(paths['converted_sessions'].sum()),
        'eligibility': 'conversion date on or after 2020-12-01; full 30-day source-window lookback',
        'interpretation': 'descriptive attribution; not causal',
        'all_channel_credits': attribution_summary,
        'credit_totals': attribution_totals,
    },
}
(DERIVED / 'findings_summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True) + '\n')
summary


{'source': {'dataset': 'bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*',
  'coverage_start': '2020-11-01',
  'coverage_end': '2021-01-31',
  'retrieved_at_utc': '2026-09-11T14:04:14.831662+00:00',
  'session_channel_definition': 'first paired event-scoped source/medium in each session'},
 'funnel': {'views': 333683,
  'engaged_sessions': 250206,
  'add_to_carts': 14919,
  'checkouts': 5956,
  'purchases': 2847,
  'engagement_rate': 0.749831,
  'add_to_cart_rate': 0.059627,
  'checkout_rate': 0.399222,
  'purchase_rate': 0.478005},
 'day_7_retention': {'eligible_cohorts': 85,
  'cohort_users': 238644,
  'retained_users': 1611,
  'rate': 0.006751,
  'definition': 'user_first_touch_timestamp within source window; day-7 complete'},
 'attribution': {'eligible_converted_sessions': 3230,
  'eligibility': 'conversion date on or after 2020-12-01; full 30-day source-window lookback',
  'interpretation': 'descriptive attribution; not causal',
  'all_channel_credits': {'first_touch'

## Findings

1. In this Google sample, 250,206 of 333,683 viewed sessions were engaged (74.98%); 14,919 engaged sessions reached cart (5.96%), 5,956 reached checkout (39.92%), and 2,847 reached purchase (47.80%). Sessions are unique `user_pseudo_id + ga_session_id` pairs and are dated by UTC session start. Source: `funnel_daily_by_channel.sql` and `findings_summary.json`.
2. Across 85 true first-touch cohorts whose day seven is fully observable, 1,611 of 238,644 users returned on day 7 (0.68%). The cohort definition uses `user_first_touch_timestamp` within the source window rather than first observed session. Source: `cohort_retention.sql` and `findings_summary.json`.
3. For 3,230 eligible converted sessions dated 2020-12-01 through 2021-01-31, Google / organic received 1,284 first-touch credits and 1,032 last-touch credits. Every included conversion has a complete 30-day source-window lookback; this is descriptive attribution, not causal. Source: `conversion_channel_paths.sql` and `findings_summary.json`.

The public sample is obfuscated. Session channels use the first paired event-scoped source/medium rather than user-acquisition `traffic_source`; unavailable paired values are explicitly labeled `(unattributed)`.